# 🎙️ Indic-Speak Bengali Text-to-Speech (Kaggle GPU Edition)

Convert Bengali text and markdown files (`.md`, `.txt`) into high-quality speech audio (`.opus`, `.wav`, or `.mp3`) using **Bodhan AI's Indic-Speak** (~3.3B parameter speech language model built on Llama-3.2-3B and Vocos neural vocoder).

---

## ⚙️ Kaggle Settings (Important!)
Before running this notebook, verify two settings in Kaggle's right-hand sidebar:
1. **Accelerator:** Set to **GPU P100** or **GPU T4 x 2** (Settings ➡️ Accelerator ➡️ GPU).
2. **Internet:** Set to **Internet On** (Settings ➡️ Internet ➡️ On) to download model weights and libraries.

---

## 🔒 One-Time Hugging Face License Requirement
The model `bodhan-ai/indic-speak` is gated under the Indic Open Model License.
1. Visit: [https://huggingface.co/bodhan-ai/indic-speak](https://huggingface.co/bodhan-ai/indic-speak)
2. Click **"Acknowledge license / Request access"** while logged into your Hugging Face account (approval is instant).
3. Add your Hugging Face token to Kaggle Secrets (`Add-ons` ➡️ `Secrets` ➡️ Key: `HF_TOKEN`) or enter it in Cell 2.

In [ ]:
# Cell 1: Environment Setup & Hardware Verification
import os
import sys
import subprocess
import torch

print('=' * 65)
print('🚀 Checking GPU Hardware...')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'✔ GPU detected: {gpu_name} ({vram_gb:.1f} GB VRAM)')
else:
    print('⚠️ No GPU detected! Please enable GPU in Kaggle (Sidebar -> Settings -> Accelerator -> GPU).')
print('=' * 65)

print('📦 Installing neural TTS & audio processing packages...')
!pip install -q "transformers>=5" snac soundfile accelerate tqdm

os.environ['HF_HUB_DISABLE_IMPLICIT_TOKEN'] = '1'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
print('✔ Environment successfully configured!')

In [ ]:
# Cell 2: Hugging Face Authentication & Model Snapshot Download
import os
from pathlib import Path
from huggingface_hub import snapshot_download

# 1. Automatically retrieve token from Kaggle Secrets if available
HF_TOKEN = ''
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    print('✔ Loaded HF_TOKEN from Kaggle Secrets.')
except Exception:
    if not HF_TOKEN:
        HF_TOKEN = os.environ.get('HF_TOKEN', '')

# If you do not use Kaggle Secrets, paste your Hugging Face token here:
# HF_TOKEN = 'hf_your_token_here'

MODEL_ID = 'bodhan-ai/indic-speak'
MODEL_DIR = '/kaggle/working/indic_speak_model'
Path(MODEL_DIR).mkdir(parents=True, exist_ok=True)

print(f'📥 Downloading \'{MODEL_ID}\' weights into {MODEL_DIR}...')
try:
    snapshot_download(
        repo_id=MODEL_ID,
        local_dir=MODEL_DIR,
        token=HF_TOKEN or None,
    )
    print('✔ Model snapshot downloaded successfully!')
except Exception as e:
    err = str(e)
    if '403' in err or 'restricted' in err.lower() or 'gated' in err.lower():
        print('\n' + '=' * 70)
        print('🔒 HUGGING FACE ACCESS REQUIRED:')
        print('The model \'bodhan-ai/indic-speak\' requires accepting the license terms.')
        print('1. Open: https://huggingface.co/bodhan-ai/indic-speak')
        print('2. Sign in and click \'Acknowledge license / Request access\'.')
        print('3. Set your token above and re-run this cell.')
        print('=' * 70 + '\n')
    raise

In [ ]:
# Cell 3: Load Indic-Speak Model onto GPU
import sys
import time

if MODEL_DIR not in sys.path:
    sys.path.insert(0, MODEL_DIR)

from inference import TTS

print('🧠 Loading Indic-Speak (3.3B) and Vocos Neural Vocoder onto GPU...')
t0 = time.time()
tts = TTS(MODEL_DIR)
print(f'✔ Indic-Speak loaded successfully in {time.time() - t0:.2f} seconds!')

In [ ]:
# Cell 4: Bengali Text Normalizer & Sentence Chunker
import re
import unicodedata
from typing import List

def normalize_bengali_text(text: str) -> str:
    """Normalizes Unicode (NFKC), strips markdown syntax, and standardizes punctuation."""
    if not text:
        return ''
    text = unicodedata.normalize('NFKC', text)
    # Strip markdown elements
    text = re.sub(r'\[([^\]]+)\]\([^)]+\)', r'\1', text)
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(r'```[\s\S]*?```', '', text)
    text = re.sub(r'`[^`]+`', '', text)
    text = re.sub(r'^\s*[#*+->]+\s*', '', text, flags=re.MULTILINE)
    text = re.sub(r'[*_~]+', '', text)
    
    # Bengali punctuation
    text = text.replace('||', '॥').replace('|', '।')
    text = re.sub(r'\?+', '?', text)
    text = re.sub(r'!+', '!', text)
    text = re.sub(r'।+', '।', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

def chunk_bengali_text(text: str, max_chunk_len: int = 350) -> List[str]:
    """
    Intelligently chunks Bengali text by sentence delimiters ('।', '॥', '?', '!', '\n').
    Groups consecutive sentences up to max_chunk_len (~20-25s of speech).
    """
    clean = normalize_bengali_text(text)
    if not clean:
        return []
    
    pattern = r'([^।॥?!\n]+[।॥?!]?)'
    raw_segments = re.findall(pattern, clean)
    sentences = [s.strip() for s in raw_segments if s.strip()]
    
    chunks = []
    current = ''
    for s in sentences:
        if len(s) > max_chunk_len:
            if current:
                chunks.append(current.strip())
                current = ''
            parts = re.split(r'([,;:\-–—]\s*)', s)
            sub_curr = ''
            for p in parts:
                if len(sub_curr) + len(p) <= max_chunk_len:
                    sub_curr += p
                else:
                    if sub_curr.strip():
                        chunks.append(sub_curr.strip())
                    sub_curr = p
            if sub_curr.strip():
                chunks.append(sub_curr.strip())
            continue
            
        candidate = f'{current} {s}'.strip() if current else s
        if len(candidate) <= max_chunk_len:
            current = candidate
        else:
            if current:
                chunks.append(current.strip())
            current = s
            
    if current:
        chunks.append(current.strip())
        
    return [c for c in chunks if c]

print('✔ Bengali text normalizer & sentence chunker ready!')

In [ ]:
# Cell 5: Synthesis Engine with Resumable Checkpoints
import os
import shutil
import subprocess
import numpy as np
import soundfile as sf
from pathlib import Path
from tqdm.auto import tqdm

SAMPLE_RATE = 24000

def synthesize_text_to_audio(
    text: str,
    output_path: str,
    speaker: str = 'Ishita',
    style: str = 'NEUTRAL',
    chunk_size: int = 350,
    pause_duration: float = 0.25,
    output_format: str = 'opus',
    checkpoint_dir: str = '/kaggle/working/chunks',
) -> dict:
    """
    Synthesizes Bengali text into audio. Saves intermediate chunks so execution is resumable!
    """
    chunks = chunk_bengali_text(text, max_chunk_len=chunk_size)
    if not chunks:
        raise ValueError('Input text has no readable content.')
        
    out_p = Path(output_path).resolve()
    out_p.parent.mkdir(parents=True, exist_ok=True)
    
    chk_dir = Path(checkpoint_dir)
    chk_dir.mkdir(parents=True, exist_ok=True)
    
    pause_samples = int(SAMPLE_RATE * pause_duration)
    silence = np.zeros(pause_samples, dtype=np.float32)
    
    generated_segments = []
    print(f'\n🎙️ Synthesizing {len(chunks)} chunk(s) (Speaker: {speaker}, Style: {style})...')
    
    for idx, chunk in enumerate(tqdm(chunks, desc='Generating Audio')):
        chunk_file = chk_dir / f'chunk_{idx:04d}.npy'
        
        # Load cached chunk if already generated (resumable!)
        if chunk_file.exists() and chunk_file.stat().st_size > 0:
            wav_np = np.load(chunk_file)
        else:
            wav = tts(
                text=chunk,
                speaker=speaker,
                style=style,
                temperature=0.6,
                top_p=0.9,
                top_k=50,
                max_new_tokens=2520,
            )
            if hasattr(wav, 'cpu'):
                wav_np = wav.cpu().numpy()
            elif hasattr(wav, 'numpy'):
                wav_np = wav.numpy()
            else:
                wav_np = np.array(wav, dtype=np.float32)
            wav_np = np.squeeze(wav_np)
            np.save(chunk_file, wav_np)
            
        generated_segments.append(wav_np)
        
    # Interleave segments with pause silence
    combined = []
    for i, seg in enumerate(generated_segments):
        combined.append(seg)
        if i < len(generated_segments) - 1 and pause_samples > 0:
            combined.append(silence)
            
    full_audio = np.concatenate(combined, axis=0)
    
    # Normalize volume
    max_val = np.max(np.abs(full_audio))
    if max_val > 0.99:
        full_audio = full_audio / max_val * 0.95
        
    temp_wav = '/kaggle/working/temp_render.wav'
    sf.write(temp_wav, full_audio, samplerate=SAMPLE_RATE)
    
    # Format conversion via ffmpeg
    fmt = output_format.lower().lstrip('.')
    if fmt == 'opus':
        cmd = ['ffmpeg', '-y', '-i', temp_wav, '-c:a', 'libopus', '-b:a', '24k', '-vbr', 'on', str(out_p)]
        subprocess.run(cmd, check=True, capture_output=True)
    elif fmt == 'mp3':
        cmd = ['ffmpeg', '-y', '-i', temp_wav, '-codec:a', 'libmp3lame', '-b:a', '64k', str(out_p)]
        subprocess.run(cmd, check=True, capture_output=True)
    else:  # wav
        shutil.copyfile(temp_wav, str(out_p))
        
    if os.path.exists(temp_wav):
        os.remove(temp_wav)
        
    dur_sec = len(full_audio) / SAMPLE_RATE
    size_kb = out_p.stat().st_size / 1024
    print(f'✔ Saved: {out_p.name} ({dur_sec:.1f}s, {size_kb:.1f} KB)')
    
    # Cleanup temporary chunk files for this document
    shutil.rmtree(chk_dir, ignore_errors=True)
    
    return {'file': str(out_p), 'duration': dur_sec, 'size_kb': size_kb}

print('✔ Checkpointed synthesis engine ready!')

In [ ]:
# Cell 6: Batch Convert All Uploaded Files
from pathlib import Path

# ==================== USER CONFIGURATION ====================
# Voice Options: 'Ishita' (Female native Bengali, default) or 'Sourav' (Male native Bengali)
SPEAKER = 'Ishita'

# Emotion / Style: 'NEUTRAL' (default), 'HAPPY', 'SAD', 'ANGER', 'FEAR'
STYLE = 'NEUTRAL'

# Output Audio Format: 'opus' (default, superior quality & compression), 'wav', or 'mp3'
OUTPUT_FORMAT = 'opus'

# Input Search Locations (Kaggle Dataset uploads or working dir)
SEARCH_DIRS = ['/kaggle/input', '/kaggle/working']
OUTPUT_DIR = '/kaggle/working/output_audio'
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
# ============================================================

# Discover files
convertible_files = []
for sdir in SEARCH_DIRS:
    for ext in ['*.txt', '*.md', '*.markdown']:
        for fp in Path(sdir).rglob(ext):
            if 'indic_speak_model' in str(fp) or 'requirements' in fp.name or 'chunks' in str(fp):
                continue
            if fp not in convertible_files:
                convertible_files.append(fp)

print('=' * 65)
print(f'📂 Found {len(convertible_files)} Bengali document(s) to convert:')
for i, f in enumerate(convertible_files, 1):
    print(f'  {i}. {f.name} ({f})')
print('=' * 65)

if not convertible_files:
    print('\n⚠️ No .txt or .md files found!')
    print('Tip: Upload your Bengali text/markdown files to /kaggle/working or add them as a Kaggle Dataset.')
else:
    results = []
    for idx, doc_path in enumerate(convertible_files, 1):
        print(f'\n[{idx}/{len(convertible_files)}] Processing: {doc_path.name}')
        content = doc_path.read_text(encoding='utf-8', errors='replace')
        
        out_file = Path(OUTPUT_DIR) / f'{doc_path.stem}.{OUTPUT_FORMAT}'
        res = synthesize_text_to_audio(
            text=content,
            output_path=str(out_file),
            speaker=SPEAKER,
            style=STYLE,
            output_format=OUTPUT_FORMAT,
        )
        results.append(res)

    print('\n' + '=' * 65)
    print(f'🎉 Conversion Complete! Successfully converted {len(results)} file(s).')
    print('=' * 65)

In [ ]:
# Cell 7: In-Notebook Audio Player & 1-Click ZIP Download
import shutil
from pathlib import Path
import IPython.display as ipd

# 1. Play audio sample directly inside Kaggle notebook
output_files = sorted(list(Path(OUTPUT_DIR).glob(f'*.{OUTPUT_FORMAT}')))
if output_files:
    sample = output_files[0]
    print(f'🎧 Playing Sample: {sample.name}')
    ipd.display(ipd.Audio(str(sample)))

# 2. Package all generated audio into a single downloadable ZIP file
zip_base = '/kaggle/working/bengali_audio_output'
zip_path = shutil.make_archive(zip_base, 'zip', OUTPUT_DIR)
zip_size_mb = Path(zip_path).stat().st_size / (1024 * 1024)

print('\n' + '=' * 65)
print(f'📦 ZIP Archive Ready for Download:')
print(f'   📁 {zip_path} ({zip_size_mb:.2f} MB)')
print('👉 Download it from the right-hand \'Output\' tab in Kaggle!')
print('=' * 65)